# 02 · Arrays and two pointers

After this notebook you can recognise the two-pointer pattern, code it without off-by-one bugs, and solve the classic array problems — pair sums, in-place dedupe, container with most water, 3Sum, merging, the Dutch flag — plus the two running-sum patterns every interviewer expects: prefix sums and Kadane's algorithm.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Big-O and timing](01_big_o_and_timing.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 2 "Arrays"

## Why this matters in interviews

- Arrays are the most common input in coding rounds, and "two pointers" is the standard way to turn an O(n²) pair search into O(n) with O(1) extra space.
- The in-place variants (dedupe, partition, merge from the back) test whether you can keep an **invariant** straight while you overwrite the array you are reading.
- Prefix sums and Kadane come back constantly in disguise: range queries, "subarray with sum k" (notebook 03), max profit, best window.

## What interviewers ask

- "Given a *sorted* array, find two numbers that add up to a target, in O(1) extra space." (LeetCode 167)
- "Remove duplicates from a sorted array in place and return the new length." (LC 26)
- "Container with most water — and why is moving the shorter line correct?" (LC 11)
- "Find all unique triplets that sum to zero. How do you avoid duplicate triplets?" (LC 15)
- "Merge two sorted arrays in place. Why fill from the back?" (LC 88)
- "Sort an array of 0s, 1s and 2s in one pass." (LC 75)
- "Answer many range-sum queries quickly." (LC 303) · "Maximum subarray sum — what if every number is negative?" (LC 53)
- Follow-ups: "Can you do it in O(1) extra space?" · "What if the input is not sorted?" (→ a hash map, [notebook 03](03_hashing_and_sliding_window.ipynb))

In [ ]:
import random
import time
from itertools import accumulate

import matplotlib.pyplot as plt
import numpy as np

random.seed(0)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})


def best_time(fn, number=1, repeat=3):
    """Seconds per call of fn(): fastest of `repeat` runs (the minimum filters out background noise)."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        best = min(best, (time.perf_counter() - t0) / number)
    return best

## The pattern: two pointers

**In plain English:** instead of checking every pair `(i, j)` — n²/2 pairs — keep two indices and move one of them each step, using what you just learned to throw away candidates that cannot be the answer. Each step discards at least one element, so the whole scan is O(n).

Two shapes cover almost every problem:

```
1) Opposite ends — sorted input, pair sums, reversing, "widest container"
      [ 1   2   4   7   11   15 ]
        L→                  ←R        sum too small → L += 1 ; too big → R -= 1

2) Same direction (reader / writer) — in-place remove, dedupe, partition
      [ 0   0   1   1   1   2   2 ]
            W   R→                    R reads every element; W marks where the next keeper goes
```

**Recognise it when you see:** a *sorted* array and a pair or triplet with a target sum; "in place" or "O(1) extra space"; reversing or palindromes; merging two sorted sequences; partitioning into groups.

**Templates:**

```python
def opposite_ends(a, target):
    lo, hi = 0, len(a) - 1
    while lo < hi:                      # strict: lo == hi would pair an element with itself
        s = a[lo] + a[hi]
        if s == target:
            return lo, hi
        if s < target:
            lo += 1                     # need bigger: only moving lo right can help
        else:
            hi -= 1                     # need smaller: only moving hi left can help
    return None

def reader_writer(a, keep):
    write = 0                           # invariant: a[:write] holds the kept elements, in order
    for read in range(len(a)):
        if keep(a[read]):
            a[write] = a[read]
            write += 1
    return write                        # new logical length
```

The helper below draws a run of an opposite-ends scan: one row per step, the two pointers coloured, and the cells already ruled out greyed.

In [ ]:
def draw_pointer_trace(values, steps, title):
    """steps: list of (lo, hi, note). Cells outside [lo, hi] are grey: they can no longer be in the answer."""
    fig, ax = plt.subplots(figsize=(10, 0.6 * len(steps) + 0.9))
    for row, (lo, hi, note) in enumerate(steps):
        y = len(steps) - 1 - row
        for i, v in enumerate(values):
            if i == lo:
                face, ink = COLORS[0], "white"
            elif i == hi:
                face, ink = COLORS[1], "white"
            elif lo < i < hi:
                face, ink = "#e7f0ff", "#0b0b0b"
            else:
                face, ink = "#f0efec", "#898781"
            ax.add_patch(plt.Rectangle((i, y), 0.92, 0.8, facecolor=face, edgecolor="none"))
            ax.text(i + 0.46, y + 0.4, str(v), ha="center", va="center", color=ink, fontsize=10)
        ax.text(len(values) + 0.3, y + 0.4, note, va="center", fontsize=9, color="#0b0b0b")
    ax.set_xlim(-0.2, len(values) + 7)
    ax.set_ylim(-0.2, len(steps))
    ax.axis("off")
    ax.set_title(title)
    plt.show()

## Problem 1 · Reverse an array in place (LC 344)

**Statement:** reverse a list of characters in place with O(1) extra memory. `["h","e","l","l","o"]` → `["o","l","l","e","h"]`.

**Brute force:** `s[::-1]` — correct, but it builds a copy: O(n) extra space. **Optimal:** swap the two ends and walk inward (opposite ends).

In [ ]:
def reverse_in_place(chars):
    """Swap the ends and walk inward. O(n) time, O(1) extra space. Returns None, like list.reverse()."""
    lo, hi = 0, len(chars) - 1
    while lo < hi:
        chars[lo], chars[hi] = chars[hi], chars[lo]
        lo += 1
        hi -= 1


s = list("hello")
reverse_in_place(s)
print("reversed:", s)

for case in [[], ["a"], ["a", "b"], list("hello"), list("abcdef")]:
    expected = case[::-1]                          # the brute-force answer (a copy)
    assert reverse_in_place(case) is None and case == expected
print("reverse_in_place: empty, single, even and odd lengths all pass")

**Complexity:** O(n) time (n/2 swaps), O(1) extra space. **Gotcha:** Python functions that mutate in place return `None` by convention (`list.sort`, `list.reverse`) — don't write `s = reverse_in_place(s)`.

## Problem 2 · Two Sum on a sorted array (LC 167)

**Statement:** `numbers` is sorted ascending. Return the 1-based indices of the two numbers that add up to `target`, using O(1) extra space. `[2, 7, 11, 15]`, target 9 → `[1, 2]`.

**Brute force:** try every pair — O(n²). **Optimal:** opposite-ends pointers. **Why it never skips the answer:** if `a[lo] + a[hi]` is too small, then `a[lo]` paired with the *largest* remaining value is already too small, so `a[lo]` cannot be in any answer — discard it (`lo += 1`). Symmetrically, if the sum is too big, `a[hi]` is out.

In [ ]:
def two_sum_sorted_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i + 1, j + 1]
    return []


def two_sum_sorted(nums, target, trace=None):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        s = nums[lo] + nums[hi]
        if s == target:
            if trace is not None:
                trace.append((lo, hi, f"{nums[lo]} + {nums[hi]} = {s}: found"))
            return [lo + 1, hi + 1]
        if trace is not None:
            move = "too small: L moves right" if s < target else "too big: R moves left"
            trace.append((lo, hi, f"{nums[lo]} + {nums[hi]} = {s}, {move}"))
        if s < target:
            lo += 1
        else:
            hi -= 1
    return []


cases = [(([2, 7, 11, 15], 9), [1, 2]), (([2, 3, 4], 6), [1, 3]), (([-1, 0], -1), [1, 2]),
         (([3, 3], 6), [1, 2]), (([1, 2, 3], 100), []), (([], 5), []), (([5], 10), [])]
for (nums, target), expected in cases:
    assert two_sum_sorted(nums, target) == expected, (nums, target)

rng = random.Random(1)          # stress test against the brute force (several answers may be valid)
for _ in range(500):
    nums = sorted(rng.randint(-20, 20) for _ in range(rng.randint(0, 12)))
    target = rng.randint(-40, 40)
    got = two_sum_sorted(nums, target)
    assert (got == []) == (two_sum_sorted_brute(nums, target) == [])
    if got:
        assert got[0] < got[1] and nums[got[0] - 1] + nums[got[1] - 1] == target
print("two_sum_sorted: 7 hand-written cases + 500 random cases agree with the brute force")

In [ ]:
trace = []
two_sum_sorted([1, 2, 4, 7, 11, 15], 15, trace)
draw_pointer_trace([1, 2, 4, 7, 11, 15], trace, "Two Sum on a sorted array, target 15: every step rules out one element")

**Complexity:** O(n) time — each step moves a pointer inward, at most n − 1 steps; O(1) extra space. If the array is *not* sorted, sorting first costs O(n log n) and loses the original indices; a hash map does it in O(n) time and O(n) space (notebook 03).

## Problem 3 · Remove duplicates from a sorted array (LC 26)

**Statement:** remove duplicates in place so each value appears once, keep the order, and return the new length `k`; only `nums[:k]` matters. `[0,0,1,1,1,2,2,3,3,4]` → `k = 5`, `nums[:5] = [0,1,2,3,4]`.

**Brute force:** `sorted(set(nums))` — O(n log n) and a new list, not in place. **Optimal:** reader/writer. Because the input is sorted, duplicates are adjacent, so "is this new?" is just "is it different from the last value I kept?".

In [ ]:
def remove_duplicates(nums):
    """In place. Invariant: nums[:write] holds each distinct value seen so far, once, in order."""
    if not nums:
        return 0
    write = 1
    for read in range(1, len(nums)):
        if nums[read] != nums[write - 1]:     # a value we have not kept yet
            nums[write] = nums[read]
            write += 1
    return write


nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(nums)
print(f"k = {k}, nums[:k] = {nums[:k]}")

for case, expected_k in [([1, 1, 2], 2), ([], 0), ([7], 1), ([2, 2, 2], 1), ([1, 2, 3], 3)]:
    distinct = sorted(set(case))
    assert remove_duplicates(case) == expected_k and case[:expected_k] == distinct
for _ in range(300):
    case = sorted(rng.randint(0, 6) for _ in range(rng.randint(0, 15)))
    distinct = sorted(set(case))
    assert case[:remove_duplicates(case)] == distinct
print("remove_duplicates: edge cases + 300 random sorted lists pass")

**Complexity:** O(n) time, O(1) extra space. The same skeleton solves "remove element" (LC 27, keep `x != val`), "move zeroes" (exercise 2) and "allow at most two copies" (LC 80, compare with `nums[write - 2]`).

## Problem 4 · Container with most water (LC 11)

**Statement:** `height[i]` is a vertical line at position `i`. Pick two lines that, with the x-axis, hold the most water: area = `min(h[i], h[j]) × (j − i)`. `[1,8,6,2,5,4,8,3,7]` → 49.

**Brute force:** every pair, O(n²). **Optimal:** start at both ends (the widest container) and **always move the shorter line**. Why: the area is capped by the shorter line. Moving the *taller* one inward shrinks the width and cannot raise that cap, so every container it could form with the shorter line is worse — the shorter line can be discarded.

In [ ]:
def max_area_brute(h):
    return max((min(h[i], h[j]) * (j - i) for i in range(len(h)) for j in range(i + 1, len(h))), default=0)


def max_area(h):
    lo, hi, best, best_pair = 0, len(h) - 1, 0, None
    while lo < hi:
        area = min(h[lo], h[hi]) * (hi - lo)
        if area > best:
            best, best_pair = area, (lo, hi)
        if h[lo] < h[hi]:
            lo += 1          # the shorter line limits every container it is part of from here on
        else:
            hi -= 1
    return best, best_pair


heights = [1, 8, 6, 2, 5, 4, 8, 3, 7]
area, (left, right) = max_area(heights)
print(f"max area = {area}, between lines {left} and {right}")

for h, expected in [([1, 8, 6, 2, 5, 4, 8, 3, 7], 49), ([1, 1], 1), ([4, 3, 2, 1, 4], 16), ([1, 2, 1], 2), ([], 0), ([5], 0)]:
    assert max_area(h)[0] == expected == max_area_brute(h), h
for _ in range(300):
    h = [rng.randint(0, 20) for _ in range(rng.randint(0, 25))]
    assert max_area(h)[0] == max_area_brute(h)
print("max_area: edge cases + 300 random inputs agree with the brute force")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(range(len(heights)), heights, width=0.25, color="#898781")
water = min(heights[left], heights[right])
ax.add_patch(plt.Rectangle((left, 0), right - left, water, color=COLORS[0], alpha=0.25, label=f"best: min({heights[left]}, {heights[right]}) x {right - left} = {area}"))
ax.bar([left, right], [heights[left], heights[right]], width=0.25, color=COLORS[0])
ax.set(title="Container with most water: the shorter of the two lines sets the water level",
       xlabel="index i", ylabel="height[i]", xticks=range(len(heights)), ylim=(0, max(heights) + 2.5))
ax.legend(loc="upper right")
plt.show()

How much does O(n) vs O(n²) matter? Time both versions as n doubles (small sizes — the brute force is the slow one):

In [ ]:
def interleaved_times(calls, number=1, rounds=2):
    """best_time for several calls, timed in alternating rounds so background load hits them all equally."""
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            best[i] = min(best[i], best_time(call, number=number, repeat=1))
    return best


sizes = [100, 200, 400, 800]
inputs = {n: [rng.randint(1, 1000) for _ in range(n)] for n in sizes}
t_brute = interleaved_times([lambda h=inputs[n]: max_area_brute(h) for n in sizes])
t_fast = interleaved_times([lambda h=inputs[n]: max_area(h) for n in sizes], number=20)
slope_brute = np.polyfit(np.log2(sizes), np.log2(t_brute), 1)[0]
slope_fast = np.polyfit(np.log2(sizes), np.log2(t_fast), 1)[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(sizes, np.array(t_brute) * 1e3, marker="o", label=f"brute force, all pairs (slope {slope_brute:.1f})")
ax.loglog(sizes, np.array(t_fast) * 1e3, marker="o", label=f"two pointers (slope {slope_fast:.1f})")
ax.set(title="Container with most water: O(n²) vs O(n)", xlabel="n (log scale)", ylabel="time, ms (log scale)")
ax.minorticks_off()
ax.set_xticks(sizes, [str(n) for n in sizes])
ax.legend()
plt.show()
print(f"at n = 800 the two-pointer version is {t_brute[-1] / t_fast[-1]:.0f}x faster")
assert slope_brute > 1.6 and slope_fast < 1.4 and t_fast[-1] < t_brute[-1]

**Complexity:** O(n) time, O(1) space. **Trap:** "move the pointer at the smaller *index*" or "move both" — only moving the shorter line is justified by the argument above. When the heights are equal, moving either is safe.

## Problem 5 · 3Sum (LC 15)

**Statement:** return all *unique* triplets `[a, b, c]` from `nums` with `a + b + c == 0`. `[-1,0,1,2,-1,-4]` → `[[-1,-1,2], [-1,0,1]]`.

**Brute force:** three nested loops and a set of sorted tuples — O(n³). **Optimal:** sort, fix the first element `nums[i]`, then solve Two Sum on the rest with opposite-ends pointers — O(n²). **Duplicates:** after sorting, equal values are adjacent, so skip an `i` equal to the previous one, and after recording a triplet move `lo` past equal values.

In [ ]:
def three_sum_brute(nums):
    found = set()
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            for k in range(j + 1, len(nums)):
                if nums[i] + nums[j] + nums[k] == 0:
                    found.add(tuple(sorted((nums[i], nums[j], nums[k]))))
    return sorted(list(t) for t in found)


def three_sum(nums):
    nums, result = sorted(nums), []
    for i in range(len(nums) - 2):
        if nums[i] > 0:
            break                                   # the smallest of the three is positive: no zero sums left
        if i > 0 and nums[i] == nums[i - 1]:
            continue                                # same first value as last time: same triplets
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            s = nums[i] + nums[lo] + nums[hi]
            if s < 0:
                lo += 1
            elif s > 0:
                hi -= 1
            else:
                result.append([nums[i], nums[lo], nums[hi]])
                lo, hi = lo + 1, hi - 1
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1                         # skip duplicate second values
    return result                                   # already in sorted order


print(three_sum([-1, 0, 1, 2, -1, -4]))
for nums, expected in [([-1, 0, 1, 2, -1, -4], [[-1, -1, 2], [-1, 0, 1]]), ([0, 1, 1], []), ([0, 0, 0], [[0, 0, 0]]),
                       ([0, 0, 0, 0], [[0, 0, 0]]), ([], []), ([-2, 0, 1, 1, 2], [[-2, 0, 2], [-2, 1, 1]])]:
    assert three_sum(nums) == expected, nums
for _ in range(300):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(0, 12))]
    assert three_sum(nums) == three_sum_brute(nums)
print("three_sum: edge cases + 300 random inputs agree with the O(n³) brute force")

**Complexity:** O(n log n) for the sort + O(n²) for the loops = **O(n²)** time; O(1) extra besides the output (Python's `sorted` makes an O(n) copy — say so). **Generalises:** k-Sum = sort + (k − 2) nested loops + two pointers → O(n^(k−1)).

## Problem 6 · Merge two sorted arrays in place (LC 88)

**Statement:** `nums1` has length `m + n`: `m` real values followed by `n` free slots. Merge the sorted `nums2` (length `n`) into it, in place. `[1,2,3,0,0,0], m=3, nums2=[2,5,6]` → `[1,2,2,3,5,6]`.

**Brute force:** copy `nums2` into the free slots and call `sort()` — O((m+n) log(m+n)). **Optimal:** three pointers, **filling from the back**. From the front you would overwrite values of `nums1` you have not read yet; the back of `nums1` is free space, and the write pointer can never overtake the `nums1` read pointer.

In [ ]:
def merge_in_place(nums1, m, nums2, n):
    i, j, write = m - 1, n - 1, m + n - 1
    while j >= 0:                       # once nums2 is used up, the rest of nums1 is already in place
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[write] = nums1[i]
            i -= 1
        else:
            nums1[write] = nums2[j]     # ties take nums2 first here, so equal nums1 values end up earlier: stable
            j -= 1
        write -= 1


a = [1, 2, 3, 0, 0, 0]
merge_in_place(a, 3, [2, 5, 6], 3)
print("merged:", a)

for n1, m, n2 in [([1], 1, []), ([0], 0, [1]), ([4, 5, 6, 0, 0, 0], 3, [1, 2, 3]), ([2, 0], 1, [1]), ([1, 1, 0, 0], 2, [1, 1])]:
    expected = sorted(n1[:m] + n2)
    merge_in_place(n1, m, n2, len(n2))
    assert n1 == expected
for _ in range(300):
    x = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))
    y = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))
    buf = x + [0] * len(y)
    merge_in_place(buf, len(x), y, len(y))
    assert buf == sorted(x + y)
print("merge_in_place: edge cases (empty sides, all-smaller, ties) + 300 random pairs pass")

**Complexity:** O(m + n) time, O(1) extra space. For k sorted *iterables* (log files, sorted chunks from disk), `heapq.merge(*iterables)` merges lazily in O(total · log k) — the heap version of this idea ([10 · Heaps](10_heaps_and_top_k.ipynb)).

## Problem 7 · Sort colours — the Dutch national flag (LC 75)

**Statement:** `nums` holds only 0, 1 and 2. Sort it in place in **one pass**. `[2,0,2,1,1,0]` → `[0,0,1,1,2,2]`.

**Brute force:** `sort()` is O(n log n); counting sort (count, then overwrite) is O(n) but two passes. **Optimal:** three pointers with the invariant

```
[ 0 0 0 | 1 1 1 | ? ? ? ? | 2 2 2 ]
          ^low    ^mid    ^high        nums[:low] = 0s, nums[low:mid] = 1s, nums[mid:high+1] unknown, nums[high+1:] = 2s
```

Look at `nums[mid]`: a 0 swaps into the 0-zone (`low` and `mid` advance), a 1 is already in place (`mid` advances), a 2 swaps to the back (`high` retreats, and `mid` stays — the value that just arrived has not been examined yet).

In [ ]:
def sort_colors(nums, trace=None):
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if trace is not None:
            trace.append((nums.copy(), low, mid, high))
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1                   # do NOT advance mid: the swapped-in value is unexamined
    if trace is not None:
        trace.append((nums.copy(), low, mid, high))


for case in [[2, 0, 2, 1, 1, 0], [2, 0, 1], [0], [], [1, 1], [2, 2, 2, 0], [0, 0, 1, 2, 2]]:
    expected = sorted(case)
    sort_colors(case)
    assert case == expected
for _ in range(300):
    case = [rng.randint(0, 2) for _ in range(rng.randint(0, 20))]
    expected = sorted(case)
    sort_colors(case)
    assert case == expected
print("sort_colors: edge cases + 300 random arrays sorted correctly in one pass")

In [ ]:
steps = []
sort_colors([2, 0, 2, 1, 1, 0], steps)
flag = {0: COLORS[7], 1: "#f0efec", 2: COLORS[0]}          # red, white, blue
fig, ax = plt.subplots(figsize=(10, 0.5 * len(steps) + 0.9))
for row, (arr, low, mid, high) in enumerate(steps):
    y = len(steps) - 1 - row
    for i, v in enumerate(arr):
        ax.add_patch(plt.Rectangle((i, y), 0.92, 0.8, facecolor=flag[v], edgecolor="none"))
        ax.text(i + 0.46, y + 0.4, str(v), ha="center", va="center", fontsize=10, color="white" if v != 1 else "#0b0b0b")
    ax.text(len(arr) + 0.3, y + 0.4, f"low={low}  mid={mid}  high={high}", va="center", fontsize=9)
ax.set_xlim(-0.2, len(steps[0][0]) + 4)
ax.set_ylim(-0.2, len(steps))
ax.axis("off")
ax.set_title("Dutch national flag: one pass, three regions growing (top row = start, bottom = done)")
plt.show()

**Complexity:** O(n) time, one pass, O(1) space. Each step either advances `mid` or retreats `high`, so the loop runs at most n times. This three-way partition is also the fix for quicksort on inputs with many duplicates ([07 · Sorting](07_sorting_algorithms.ipynb)).

## Problem 8 · Prefix sums for range-sum queries (LC 303)

**In plain English:** precompute running totals once; then any range total is the difference of two running totals — like reading two odometer values instead of re-driving the road.

**Statement:** answer many queries `sum(nums[l..r])` on a fixed array. **Brute force:** sum the slice per query — O(n) each, O(n·q) total. **Optimal:** `prefix[i] = sum(nums[:i])` (with `prefix[0] = 0`), then `sum(nums[l..r]) = prefix[r+1] − prefix[l]`: O(n) once, O(1) per query.

In [ ]:
class RangeSum:
    def __init__(self, nums):
        self.prefix = list(accumulate(nums, initial=0))    # prefix[i] = sum(nums[:i]); len = n + 1

    def query(self, l, r):
        """Sum of nums[l..r], inclusive, in O(1)."""
        return self.prefix[r + 1] - self.prefix[l]


example = RangeSum([-2, 0, 3, -5, 2, -1])                 # the LeetCode example
print("sum(0..2) =", example.query(0, 2), " sum(2..5) =", example.query(2, 5), " sum(0..5) =", example.query(0, 5))
assert (example.query(0, 2), example.query(2, 5), example.query(0, 5)) == (1, -1, -3)

values = [rng.randint(-9, 9) for _ in range(200)]
rs = RangeSum(values)
for _ in range(1_000):
    l = rng.randrange(len(values))
    r = rng.randrange(l, len(values))
    assert rs.query(l, r) == sum(values[l:r + 1])
assert rs.query(0, 0) == values[0] and rs.query(0, len(values) - 1) == sum(values)
print("RangeSum: LeetCode example + 1,000 random queries match the brute force")

In [ ]:
nums = [3, -1, 4, 1, -5, 9, 2, -6, 5, 3]
prefix = list(accumulate(nums, initial=0))
l, r = 2, 6
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(range(len(nums)), nums, width=0.6, color=[COLORS[1] if l <= i <= r else "#c3c2b7" for i in range(len(nums))],
       label=f"nums (highlighted: indices {l}..{r})")
ax.plot(np.arange(len(prefix)) - 0.5, prefix, marker="o", color=COLORS[0], label="prefix[i] = sum(nums[:i]), drawn at the gap before i")
ax.annotate("", xy=(r + 0.5, prefix[r + 1]), xytext=(r + 0.5, prefix[l]), arrowprops=dict(arrowstyle="<->", lw=1.6))
ax.plot([l - 0.5, r + 0.5], [prefix[l], prefix[l]], color="#898781", lw=1)
ax.text(7.5, -4.5, f"arrow: prefix[{r + 1}] - prefix[{l}]\n= {prefix[r + 1]} - {prefix[l]} = {prefix[r + 1] - prefix[l]}", va="center", fontsize=8)
ax.set(title=f"Range sum = difference of two prefix sums: sum(nums[{l}..{r}]) = {sum(nums[l:r + 1])}",
       xlabel="index", ylabel="value", xticks=range(len(nums)))
ax.legend(loc="upper left", fontsize=8)
plt.show()
assert prefix[r + 1] - prefix[l] == sum(nums[l:r + 1])

**Complexity:** O(n) preprocessing and space, O(1) per query. **Variations:** 2-D prefix sums for rectangle queries (`P[r][c]` with inclusion–exclusion); prefix sums + a hash map count "subarrays with sum k" even with negative numbers ([notebook 03](03_hashing_and_sliding_window.ipynb)). If the array changes between queries, use a Fenwick / segment tree (O(log n) update and query).

## Problem 9 · Maximum subarray — Kadane's algorithm (LC 53)

**Statement:** find the contiguous, non-empty subarray with the largest sum. `[-2,1,-3,4,-1,2,1,-5,4]` → 6 (`[4,-1,2,1]`).

**Brute force:** every start `i`, extend `j` with a running sum — O(n²). **Kadane, in plain English:** walk left to right keeping the best sum of a subarray *ending here*. If the running sum so far is negative it can only drag the next element down, so drop it and start fresh. The answer is the best of those running sums.

In [ ]:
def max_subarray_brute(nums):
    best = nums[0]
    for i in range(len(nums)):
        running = 0
        for j in range(i, len(nums)):
            running += nums[j]
            best = max(best, running)
    return best


def max_subarray(nums):
    """Kadane. Returns (best sum, (start, end)) — O(n) time, O(1) space."""
    best = current = nums[0]
    start = best_start = best_end = 0
    for i in range(1, len(nums)):
        if current < 0:                         # a negative prefix only hurts: start fresh at i
            current, start = nums[i], i
        else:
            current += nums[i]
        if current > best:
            best, best_start, best_end = current, start, i
    return best, (best_start, best_end)


best, (i0, i1) = max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4])
print(f"best sum {best} from index {i0} to {i1}")
for nums, expected in [([-2, 1, -3, 4, -1, 2, 1, -5, 4], 6), ([1], 1), ([5, 4, -1, 7, 8], 23), ([-3, -1, -2], -1), ([0], 0), ([-1, 0, -2], 0)]:
    assert max_subarray(nums)[0] == expected == max_subarray_brute(nums), nums
for _ in range(500):
    nums = [rng.randint(-10, 10) for _ in range(rng.randint(1, 20))]
    got, (a, b) = max_subarray(nums)
    assert got == max_subarray_brute(nums) == sum(nums[a:b + 1])
print("max_subarray: edge cases (incl. all-negative) + 500 random arrays agree with the brute force")

In [ ]:
nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
current_line, best_line, current, best = [], [], 0, float("-inf")
for i, x in enumerate(nums):
    current = x if i == 0 or current < 0 else current + x
    best = max(best, current)
    current_line.append(current)
    best_line.append(best)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.axvspan(i0 - 0.4, i1 + 0.4, color=COLORS[2], alpha=0.15, label=f"best subarray, indices {i0}..{i1}")
ax.bar(range(len(nums)), nums, width=0.5, color="#c3c2b7", label="nums[i]")
ax.plot(current_line, marker="o", color=COLORS[0], label="best sum ending at i (resets when it was negative)")
ax.step(range(len(nums)), best_line, where="post", color=COLORS[1], label="best sum so far")
ax.axhline(0, color="#898781", lw=1)
ax.set(title=f"Kadane's running sums on {nums}", xlabel="index i", ylabel="sum", xticks=range(len(nums)),
       ylim=(min(nums) - 1, max(best_line) + 4))      # headroom above the data for the legend
ax.legend(loc="upper left", fontsize=8, ncol=2)
plt.show()

Read the chart: the blue line drops to −2 at index 2, so at index 3 Kadane starts fresh (4 alone beats −2 + 4); from there it climbs to 6 at index 6, and the orange "best so far" never beats that.

**Complexity:** O(n) time, O(1) space. **Traps:** initialise with `nums[0]`, not 0 — otherwise an all-negative array wrongly returns 0; the subarray must be non-empty. **Same idea, other clothes:** max profit from one buy and one sell (exercise 4) is Kadane on the day-to-day price changes, or equivalently "prefix sum minus the smallest earlier prefix sum".

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| sorted array + pair with a target sum | opposite-ends pointers, O(n), O(1) space |
| unsorted array + pair with a target sum | hash map of seen values ([notebook 03](03_hashing_and_sliding_window.ipynb)) |
| triplets / k-sum | sort + fixed outer loop(s) + two pointers; skip equal neighbours to dedupe |
| "in place", "return the new length", remove / dedupe / move | reader/writer pointers |
| widest / largest area between two lines | opposite ends, move the shorter side |
| merge two sorted arrays with free space at the end | three pointers, fill from the back |
| three categories (0/1/2, low/mid/high) | Dutch national flag: `low`, `mid`, `high` |
| many range-sum queries on a fixed array | prefix sums: `P[r+1] − P[l]` |
| max / min sum of a contiguous subarray | Kadane (reset when the running sum goes negative) |
| palindrome / reverse | opposite ends, compare or swap |

## Try it yourself

**1. Valid palindrome (LC 125).** Return True if a string is a palindrome after keeping only letters and digits and ignoring case: `"A man, a plan, a canal: Panama"` → True, `"race a car"` → False. Use O(1) extra space (no cleaned copy).

In [ ]:
# Solution — try it yourself first, then run this
def is_palindrome(s):
    lo, hi = 0, len(s) - 1
    while lo < hi:
        if not s[lo].isalnum():
            lo += 1
        elif not s[hi].isalnum():
            hi -= 1
        elif s[lo].lower() != s[hi].lower():
            return False
        else:
            lo, hi = lo + 1, hi - 1
    return True


for text, expected in [("A man, a plan, a canal: Panama", True), ("race a car", False), ("", True), (" ", True),
                       ("0P", False), ("ab_a", True), ("No 'x' in Nixon", True)]:
    assert is_palindrome(text) == expected, text
    cleaned = [c.lower() for c in text if c.isalnum()]
    assert expected == (cleaned == cleaned[::-1])      # the O(n)-space brute force agrees
print("is_palindrome: all cases pass")

**2. Move zeroes (LC 283).** Move all 0s to the end in place, keeping the order of the other elements: `[0,1,0,3,12]` → `[1,3,12,0,0]`.

In [ ]:
# Solution — try it yourself first, then run this
def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write], nums[read] = nums[read], nums[write]    # swap keeps every value; zeros drift right
            write += 1


for case in [[0, 1, 0, 3, 12], [0], [1], [], [0, 0, 1], [1, 2, 3], [4, 0, 0, 5, 0, 6]]:
    expected = [x for x in case if x != 0] + [0] * case.count(0)
    move_zeroes(case)
    assert case == expected
print("move_zeroes: all cases pass, order of non-zeros preserved")

**3. Squares of a sorted array (LC 977).** `nums` is sorted and may contain negatives; return the sorted squares in O(n): `[-4,-1,0,3,10]` → `[0,1,9,16,100]`. (Hint: the largest square is at one of the two ends.)

In [ ]:
# Solution — try it yourself first, then run this
def sorted_squares(nums):
    result = [0] * len(nums)
    lo, hi = 0, len(nums) - 1
    for write in range(len(nums) - 1, -1, -1):     # fill from the back with the larger end
        if abs(nums[lo]) > abs(nums[hi]):
            result[write] = nums[lo] ** 2
            lo += 1
        else:
            result[write] = nums[hi] ** 2
            hi -= 1
    return result


for case in [[-4, -1, 0, 3, 10], [-7, -3, 2, 3, 11], [], [5], [-5], [-3, -2, -1]]:
    assert sorted_squares(case) == sorted(x * x for x in case), case
for _ in range(200):
    case = sorted(rng.randint(-20, 20) for _ in range(rng.randint(0, 15)))
    assert sorted_squares(case) == sorted(x * x for x in case)
print("sorted_squares: O(n) two-pointer result matches the O(n log n) brute force")

**4. Best time to buy and sell stock (LC 121).** Given daily prices, return the maximum profit from one buy followed by one sell (0 if none): `[7,1,5,3,6,4]` → 5. Do it in one pass.

In [ ]:
# Solution — try it yourself first, then run this
def max_profit(prices):
    lowest, best = float("inf"), 0
    for p in prices:
        lowest = min(lowest, p)          # cheapest buy so far (a running minimum, like Kadane's reset)
        best = max(best, p - lowest)     # best sale today
    return best


def max_profit_brute(prices):
    return max((prices[j] - prices[i] for i in range(len(prices)) for j in range(i + 1, len(prices))), default=0) if prices else 0


for prices, expected in [([7, 1, 5, 3, 6, 4], 5), ([7, 6, 4, 3, 1], 0), ([], 0), ([3], 0), ([2, 4, 1], 2), ([1, 2], 1)]:
    assert max_profit(prices) == expected == max(0, max_profit_brute(prices)), prices
for _ in range(300):
    prices = [rng.randint(1, 30) for _ in range(rng.randint(0, 15))]
    assert max_profit(prices) == max(0, max_profit_brute(prices))
print("max_profit: all cases pass")

## Say it in an interview

- **30-second answer:** "Two pointers replaces a pair search with a linear scan: two indices move monotonically, and each move is justified because it discards a candidate that cannot be the answer. It is O(n) time and O(1) extra space. Opposite ends for sorted pair sums and containers; reader/writer for in-place filtering; three pointers for merges and the Dutch flag."
- **State the invariant** before coding: "everything left of `write` is the answer so far", "`nums[:low]` are 0s". Interviewers reward it, and it prevents off-by-one bugs.
- **Justify the move:** "the sum is too small, so `a[lo]` paired with the largest remaining value is already too small — `a[lo]` is out."
- **Key facts:** 3Sum is O(n²) (sort + n two-pointer scans) and needs duplicate skipping; merge in place fills from the back; prefix sums are O(n) build, O(1) query; Kadane is O(n), O(1).
- **Follow-ups:** "Unsorted input?" — hash map, O(n) space (or sort for O(n log n)). "Stream / too big for memory?" — prefix sums and Kadane work in one pass; opposite ends needs random access. "Array changes between queries?" — Fenwick/segment tree.
- **Traps:** `while lo < hi` vs `lo <= hi` (can an element pair with itself?); forgetting to skip duplicates in 3Sum; not advancing `mid` after swapping a 2 in the Dutch flag; Kadane initialised to 0 (fails on all-negative input); `nums[::-1]` and slices are O(n) copies, not O(1) space.

## Next

- [03 · Hashing and sliding window](03_hashing_and_sliding_window.ipynb) — two sum on *unsorted* data, subarray sum = k, windows
- [06 · Binary search](06_binary_search.ipynb) — the other way to exploit a sorted array
- [07 · Sorting algorithms](07_sorting_algorithms.ipynb) — "sort first" is often what makes two pointers possible
- [03 · List and dict problems](../02_python_problems/03_list_and_dict_problems.ipynb) — more warm-up practice
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 2, "Arrays")